# Imports 

In [3]:
import pandas as pd
import numpy as np
import os

import seaborn

## repeated printouts
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Read in data and make sure relevant columns are string/character 

- San Diego data: `naics_code` and `account_key`
- NAICS details data: `naics` (North American Industry Classification Systems)

Run code below; if pulling from github, pathname should be fine; if working elsewhere may need to edit path name at read in 

In [4]:
# uncomment if u need to find your path: os.getcwd()
SD = sd_df = pd.read_csv("../public_data/sd_df.csv")
NA = naics_df = pd.read_csv("../public_data/naics_df.csv")

In [5]:
cols_sd_use = ["naics_code", "account_key"]
cols_naics_use = ["naics"]

sd_df[cols_sd_use] = sd_df[cols_sd_use].astype(str)
naics_df[cols_naics_use] = naics_df[cols_naics_use].astype(str)

sd_df.dtypes
naics_df.dtypes
sd_df.head()
naics_df.head()

account_key          object
dba_name             object
council_district     object
naics_code           object
naics_description    object
naics_nchar           int64
dtype: object

naics                object
naics_description    object
dtype: object

,account_key,dba_name,council_district,naics_code,naics_description,naics_nchar
0,1974000448,ERNST & YOUNG LLP,cd_1,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6
1,1974011093,HECHT SOLBERG ROBINSON GOLDBERG & BAGLEY LLP,cd_3,5411,LEGAL SERVICES,4
2,1978039819,RSM US LLP,cd_1,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6
3,1978042092,THORSNES BARTOLOTTA MCGUIRE LLP,cd_3,5411,LEGAL SERVICES,4
4,1979046817,KORENIC & WOJDOWSKI LLP,cd_7,5412,ACCOUNTING/TAX PREP/BOOKKEEP/PAYROLL SERVICES,4


,naics,naics_description
0,111140,Wheat Farming
1,111160,Rice Farming
2,111150,Corn Farming
3,111110,Soybean Farming
4,111120,Oilseed (except Soybean) Farming


## "Inner join"- retain only San Diego businesses with details on their NAICS code

- Use the `naics_code` column in the San Diego business data as the join key
- Use the `naics` column in the NAICS code details data as the join key

- Do an inner join of the San Diego data onto the NAICS code details using these join keys
- After the inner join, print some examples of San Diego businesses lost in the merge
- Use value_counts() on the `naics_nchar` column in the San Diego data to see why they might have gotten lost


In [9]:
# do the inner join
SD_merged = SD.merge(
    naics_df,
    left_on="naics_code",
    right_on="naics",
    how="inner"
)

# look at result
SD_merged.head()

# businesses whose naics_code did not find a match in naics_df
lost_sd = sd_df.loc[~sd_df["naics_code"].isin(SD_merged["naics"])]

# print some examples
lost_sd[["account_key", "dba_name", "naics_code", "naics_nchar"]].head(10)

sd_df["naics_nchar"].value_counts()

,account_key,dba_name,council_district,naics_code,naics_description_x,naics_nchar,naics,naics_description_y
0,1974000448,ERNST & YOUNG LLP,cd_1,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6,541211,Offices of Certified Public Accountants
1,1978039819,RSM US LLP,cd_1,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6,541211,Offices of Certified Public Accountants
2,1982006408,MAX L. PERLATTI & ASSOCIATES LLP,cd_2,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6,541211,Offices of Certified Public Accountants
3,1984014611,RBTK LLP,cd_6,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6,541211,Offices of Certified Public Accountants
4,1985008871,CWGB&S CPAS,cd_3,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6,541211,Offices of Certified Public Accountants


,account_key,dba_name,naics_code,naics_nchar
1,1974011093,HECHT SOLBERG ROBINSON GOLDBERG & BAGLEY LLP,5411,4
3,1978042092,THORSNES BARTOLOTTA MCGUIRE LLP,5411,4
4,1979046817,KORENIC & WOJDOWSKI LLP,5412,4
5,1979053082,GRIMM VRANJES GREER STEPHAN & BRIDGMAN LLP,5411,4
6,1981000840,BENINK & SLAVENS LLP,5411,4
10,1987006330,FINCH THORTON & BAIRD LLP,5411,4
11,1987007978,FINKELSTEIN & KRINSK LLP,5411,4
12,1988000382,MANHATTAN,7221,4
13,1989013971,TIC TAC TOW LLP,48841,5
14,1990012245,LATHAM & WATKINS LLP,54111,5


naics_nchar
5    150
4     89
6     49
3     42
2     23
Name: count, dtype: int64

## "Left join"- retain all sd businesses even if naics code isn't in `naics_df`

- Using the same join keys as above, and treating the San Diego businesses as the left hand side data, left join the naics code details onto the San Diego businesses
- Use the `indicator` argument within merge to create an indicator, `naics_merge_status`, to help with later merge diagnostics and examine sample of ones that didn't merge
- Use the `suffixes` argument within merge to add `_sd` as the left suffix, `_census` as the right suffix
- Use `naics_merge_status` in the merged result to look at a sample of San Diego businesses that weren't matched with `naics_df`

In [12]:
SD_left = SD.merge(
    naics_df,
    left_on="naics_code",
    right_on="naics",
    how="left",
    indicator="naics_merge_status",
    suffixes=("_sd", "_census")
)

SD_left.head()

SD_left["naics_merge_status"].value_counts()

SD_left.loc[
    SD_left["naics_merge_status"] == "left_only",
    ["account_key", "dba_name", "naics_code", "naics_nchar", "naics_description_sd", "naics_description_census"]
].head(10)

,account_key,dba_name,council_district,naics_code,naics_description_sd,naics_nchar,naics,naics_description_census,naics_merge_status
0,1974000448,ERNST & YOUNG LLP,cd_1,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6,541211,Offices of Certified Public Accountants,both
1,1974011093,HECHT SOLBERG ROBINSON GOLDBERG & BAGLEY LLP,cd_3,5411,LEGAL SERVICES,4,NaN,NaN,left_only
2,1978039819,RSM US LLP,cd_1,541211,OFFICES OF CERTIFIED PUBLIC ACCOUNTANTS,6,541211,Offices of Certified Public Accountants,both
3,1978042092,THORSNES BARTOLOTTA MCGUIRE LLP,cd_3,5411,LEGAL SERVICES,4,NaN,NaN,left_only
4,1979046817,KORENIC & WOJDOWSKI LLP,cd_7,5412,ACCOUNTING/TAX PREP/BOOKKEEP/PAYROLL SERVICES,4,NaN,NaN,left_only


naics_merge_status
left_only     318
both           52
right_only      0
Name: count, dtype: int64

,account_key,dba_name,naics_code,naics_nchar,naics_description_sd,naics_description_census
1,1974011093,HECHT SOLBERG ROBINSON GOLDBERG & BAGLEY LLP,5411,4,LEGAL SERVICES,NaN
3,1978042092,THORSNES BARTOLOTTA MCGUIRE LLP,5411,4,LEGAL SERVICES,NaN
4,1979046817,KORENIC & WOJDOWSKI LLP,5412,4,ACCOUNTING/TAX PREP/BOOKKEEP/PAYROLL SERVICES,NaN
5,1979053082,GRIMM VRANJES GREER STEPHAN & BRIDGMAN LLP,5411,4,LEGAL SERVICES,NaN
6,1981000840,BENINK & SLAVENS LLP,5411,4,LEGAL SERVICES,NaN
10,1987006330,FINCH THORTON & BAIRD LLP,5411,4,LEGAL SERVICES,NaN
11,1987007978,FINKELSTEIN & KRINSK LLP,5411,4,LEGAL SERVICES,NaN
12,1988000382,MANHATTAN,7221,4,FULL-SERVICE RESTAURANTS,NaN
13,1989013971,TIC TAC TOW LLP,48841,5,MOTOR VEHICLE TOWING,NaN
14,1990012245,LATHAM & WATKINS LLP,54111,5,OFFICES OF LAWYERS,NaN


## Use group by and agg to see if there are differences in merge rates by area

- Using the left-joined dataframe created in previous step, create a boolean indicator, `is_lost`, that equals `True` if the merge indicator is equal to "left_only" (and is otherwise false)
- Find the proportion of businesses by council district that were lost in the left join (SD businesses that failed to match to `naics_df`). To do this, group by `council_district` and use the shortcut of taking the mean of the `is_lost` indicator to find the proportions lost by `council_district`. 

In [13]:
# indicator for businesses lost in the merge
SD_left["is_lost"] = SD_left["naics_merge_status"] == "left_only"

# proportion lost by council district
lost_by_district = (
    SD_left
    .groupby("council_district")
    .agg(prop_lost=("is_lost", "mean"))
    .sort_values("prop_lost", ascending=False)
)

lost_by_district

lost_by_district = (
    SD_left
    .groupby("council_district")
    .agg(
        prop_lost=("is_lost", "mean"),
        n_businesses=("account_key", "count"),
        n_lost=("is_lost", "sum")
    )
    .sort_values("prop_lost", ascending=False)
)

lost_by_district

,prop_lost
council_district,
cd_4,1.000000
cd_9,1.000000
cd_8,0.933333
cd_7,0.918919
cd_5,0.916667
cd_1,0.880597
cd_6,0.859649
cd_3,0.812500
cd_2,0.804348


,prop_lost,n_businesses,n_lost
council_district,,,
cd_4,1.000000,4,4
cd_9,1.000000,8,8
cd_8,0.933333,15,14
cd_7,0.918919,37,34
cd_5,0.916667,24,22
cd_1,0.880597,67,59
cd_6,0.859649,57,49
cd_3,0.812500,112,91
cd_2,0.804348,46,37


## Optional challenge exercise: add lagging 0's and see if merge rate improves

You noticed earlier that a big reason for non-matches is that the San Diego tax certificate NAICS codes were often less than six digits long, while the Census ones were always 6 digits.

You wonder if this is an issue where 0's in some of the SD's data naics codes got cut off (eg 540000 became 54), and if so whether adding these lagging zeros would improve the merge rate in a left join.

- Pad the `naics_code` column in `sd_df` with 0's to get that column up to 6-digits, using one of two approaches: 
  1. `str.pad` in pandas (https://pandas.pydata.org/docs/reference/api/pandas.Series.str.pad.html)
  2. for more of a challenge, write your own function! It should check the # of digits in the naics code string and pad it with 0's at the end up to 6 digits. To execute your function, use row-wise apply: `df.apply(lambda row: funcname(row.column), axis=1)`.
- Perform the same left join as above and see how the match rate changes
- Create an indicator variable--`is_new_match`---for new matches under the padded NAICS code; compare the `naics_description` column from San Diego versus Census in the new dataset for a sample of these new matches and comment on whether the padding seems to be correct

In [15]:

sd_padded = sd_df.copy()

# pad naics_code with trailing zeros up to 6 digits
sd_padded["naics_code_padded"] = sd_padded["naics_code"].str.pad(
    width=6,
    side="right",
    fillchar="0"
)

# redo left join using padded NAICS code
SD_left_padded = sd_padded.merge(
    naics_df,
    left_on="naics_code_padded",
    right_on="naics",
    how="left",
    indicator="naics_merge_status",
    suffixes=("_sd", "_census")
)

# compare old vs new match rates
old_match_rate = (SD_left["naics_merge_status"] == "both").mean()
new_match_rate = (SD_left_padded["naics_merge_status"] == "both").mean()

print("Old match rate:", old_match_rate)
print("New match rate:", new_match_rate)
print("Improvement:", new_match_rate - old_match_rate)

# create indicators
old_matched = (SD_left["naics_merge_status"] == "both").fillna(False)
new_matched = (SD_left_padded["naics_merge_status"] == "both").fillna(False)

SD_left_padded["is_new_match"] = (~old_matched.to_numpy()) & (new_matched.to_numpy())

print("\nNumber of new matches:")
print(SD_left_padded["is_new_match"].value_counts())

# inspect a sample of the new matches
print("\nSample of newly matched businesses:")
print(
    SD_left_padded.loc[
        SD_left_padded["is_new_match"],
        [
            "account_key",
            "dba_name",
            "naics_code",
            "naics_code_padded",
            "naics_description_sd",
            "naics_description_census"
        ]
    ].head(15)
)

Old match rate: 0.14054054054054055
New match rate: 0.531390134529148
Improvement: 0.3908495939886074


ValueError: operands could not be broadcast together with shapes (370,) (446,) 